In [ ]:
import nltk
import os
from nltk.corpus import gutenberg, stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.tag import PerceptronTagger

# 1. 경로 설정
nltk_path = os.path.join(os.path.expanduser("~"), "nltk_data")
if nltk_path not in nltk.data.path:
    nltk.data.path.append(nltk_path)

# 2. 최신 버전에서 요구하는 리소스 이름으로 다운로드
# 에러 메시지에서 명시한 'averaged_perceptron_tagger_eng'를 다운로드합니다.
nltk.download('averaged_perceptron_tagger_eng', download_dir=nltk_path)
nltk.download('stopwords', download_dir=nltk_path)
nltk.download('wordnet', download_dir=nltk_path)
nltk.download('punkt', download_dir=nltk_path)

# 3. 태거 객체 수동 로드 (경로 직접 지정)
# 검색 결과 확인된 경로를 사용합니다.
tagger_path = os.path.join(nltk_path, 'taggers/averaged_perceptron_tagger/averaged_perceptron_tagger.pickle')
tagger = PerceptronTagger(load=tagger_path)

# 4. 데이터 및 도구 초기화
alice_raw = gutenberg.raw('carroll-alice.txt')
wnl = WordNetLemmatizer()
sw = set(stopwords.words("english"))

wc = {}

# 5. 분석 실행
for line in alice_raw.splitlines():
    if not line.strip():
        continue
    
    tokens = word_tokenize(line)
    # 직접 생성한 태거를 사용하여 분석
    tagged_tokens = tagger.tag(tokens)
    
    for word, wordtype in tagged_tokens:
        word = word.lower()
        if wordtype.startswith("V") and word not in sw and word.isalpha():
            lemma = wnl.lemmatize(word, wordnet.VERB)
            wc[lemma] = wc.get(lemma, 0) + 1

# 6. 결과 출력 (상위 20개)
sorted_wc = sorted(wc.items(), key=lambda x: x[1], reverse=True)
for k, v in sorted_wc[:20]:
    print(f"{k}: {v}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 데이터 준비 (상위 15개 추출)
sorted_wc = sorted(wc.items(), key=lambda x: x[1], reverse=True)[:15]
words = [item[0] for item in sorted_wc]
counts = [item[1] for item in sorted_wc]

# 2. 시각화 설정
x_pos = np.arange(len(words)) # numpy를 이용한 x축 위치 설정

plt.figure(figsize=(12, 6))
plt.bar(x_pos, counts, align='center', color='skyblue', edgecolor='navy')

# 3. 축 및 제목 설정
plt.xticks(x_pos, words, rotation=45)
plt.xlabel('Verbs')
plt.ylabel('Frequency')
plt.title('Top 15 Verbs in Alice\'s Adventures in Wonderland')

# 4. 수치 표시 (막대 위에 빈도수 출력)
for i, v in enumerate(counts):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show() # 노트북 환경에서 즉시 확인
# plt.savefig('verb_frequency.png') # 파일 저장이 필요한 경우 사용